In [75]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load Data

In [76]:
messages = pd.read_csv(
    r"C:\Users\ankit\Downloads\SMSSpamCollection.txt",
    sep='\t',
    header=None
)

messages.columns = ['label', 'message']

# Preprocessing

In [77]:
nltk.download('stopwords')

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    review = re.sub('[^a-zA-Z]', ' ', text)
    review = review.lower().split()
    review = [ps.stem(word) for word in review if word not in stop_words]
    return ' '.join(review)

corpus = messages['message'].apply(preprocess)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ankit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Train-Test Split (common for all models)

In [78]:
y = pd.get_dummies(messages['label'], drop_first=True).values.ravel()

X_train, X_test, y_train, y_test = train_test_split(
    corpus, y, test_size=0.2, random_state=42
)

# Feature Engineering

## Bag of Words

In [79]:
cv = CountVectorizer(max_features=5000)
X_train_bow = cv.fit_transform(X_train)
X_test_bow = cv.transform(X_test)

## TF-IDF

In [80]:
final_vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)

final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_vec, y_train)
TfidfVectorizer(ngram_range=(1,2), max_features=5000)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


# Models

In [81]:
nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)

# Train + Compare Models

In [82]:
results = {}

# --- BOW ---
nb.fit(X_train_bow, y_train)
results["BOW - Naive Bayes"] = accuracy_score(y_test, nb.predict(X_test_bow))

lr.fit(X_train_bow, y_train)
results["BOW - Logistic Regression"] = accuracy_score(y_test, lr.predict(X_test_bow))

# --- TF-IDF ---
nb.fit(X_train_tfidf, y_train)
results["TFIDF - Naive Bayes"] = accuracy_score(y_test, nb.predict(X_test_tfidf))

lr.fit(X_train_tfidf, y_train)
results["TFIDF - Logistic Regression"] = accuracy_score(y_test, lr.predict(X_test_tfidf))

In [83]:
print("\n📊 MODEL COMPARISON RESULTS:\n")

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")


📊 MODEL COMPARISON RESULTS:

BOW - Naive Bayes: 0.9830
BOW - Logistic Regression: 0.9839
TFIDF - Naive Bayes: 0.9722
TFIDF - Logistic Regression: 0.9722


# Best Model choose

In [84]:
best_model_name = max(results, key=results.get)
print("\n🏆 Best Model:", best_model_name)


🏆 Best Model: BOW - Logistic Regression


# LIVE CUSTOMER MESSAGE TESTING

In [85]:
final_model = lr
final_vectorizer = tfidf

In [86]:
def predict_spam(text):
    cleaned = preprocess(text)
    vector = final_vectorizer.transform([cleaned])
    result = final_model.predict(vector)
    
    return "🚨 SPAM" if result[0] == 1 else "✅ HAM"

In [87]:
print(predict_spam("Congratulations! You won a free iPhone"))
print(predict_spam("Hey, are we meeting today?"))
print(predict_spam("URGENT! Claim your reward now"))
print(predict_spam("Can you share the report?"))

✅ HAM
✅ HAM
🚨 SPAM
✅ HAM
